# Experiment 2 — Data Preprocessing Using Python

## Step 1 — Upload and Import the Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose application_matching_dataset.csv (or .xlsx) from your computer

In [ ]:
import pandas as pd
import numpy as np

# If using the CSV:
df = pd.read_csv('application_matching_dataset.csv')

# If using the Excel file instead, use this line rather than the one above:
# df = pd.read_excel('application_matching_dataset.xlsx')

print("Shape:", df.shape)
df.head()

## Step 2 — Initial Exploration

In [ ]:
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
df.describe(include='all').T

In [ ]:
print(df['branch'].unique())
print(df['city'].unique())
print(df['github_linked'].unique())
print(df['application_date'].sample(5, random_state=1).tolist())

## Step 3 — Standardize Inconsistent Categorical Text

In [ ]:
for col in ['branch', 'city']:
    df[col] = df[col].str.strip().str.title()

branch_map = {
    'Comp Engg': 'Computer Engineering', 'It': 'Information Technology',
    'Info Tech': 'Information Technology', 'Entc': 'Electronics & Telecom',
    'E&Tc': 'Electronics & Telecom', 'Mech Engg': 'Mechanical Engineering',
    'Civil': 'Civil Engineering', 'Ai & Ds': 'AI & Data Science',
    'Ai/Ds': 'AI & Data Science', 'Ai & Data Science': 'AI & Data Science'
}
df['branch'] = df['branch'].replace(branch_map)

print("Unique branches:", df['branch'].nunique(), "| Unique cities:", df['city'].nunique())

## Step 4 — Parse Mixed Date Formats

In [ ]:
df['application_date'] = pd.to_datetime(
    df['application_date'], format='mixed', dayfirst=True, errors='coerce'
)
print("Rows where date failed to parse:", df['application_date'].isnull().sum())

# Now that it's a real datetime, useful time features can be extracted:
df['application_month'] = df['application_date'].dt.month
df['application_quarter'] = df['application_date'].dt.quarter

## Step 5 — Clean the Mixed-Type Numeric Column

In [ ]:
df['expected_stipend'] = pd.to_numeric(df['expected_stipend'], errors='coerce')
print("Stipend values that became missing (were text placeholders):", df['expected_stipend'].isnull().sum())

## Step 6 — Standardize the Inconsistent Boolean Column

In [ ]:
bool_map = {'yes': 1, 'y': 1, '1': 1, 'no': 0, 'n': 0, '0': 0}
df['github_linked'] = df['github_linked'].str.lower().map(bool_map)
print(df['github_linked'].value_counts(dropna=False))

## Step 7 — Correct Invalid / Impossible Values

In [ ]:
# CGPA is on a 0-10 scale — anything above 10 is a data entry error
df.loc[df['cgpa'] > 10, 'cgpa'] = np.nan

# skill_gap_count is a count — it cannot be negative
df.loc[df['skill_gap_count'] < 0, 'skill_gap_count'] = np.nan

## Step 8 — Handle Missing Values

In [ ]:
for col in ['cgpa', 'match_score', 'time_to_response_days', 'expected_stipend', 'skill_gap_count']:
    df[col] = df[col].fillna(df[col].median())

df['city'] = df['city'].fillna(df['city'].mode()[0])
df['required_skills'] = df['required_skills'].fillna('Unknown')
# github_username is intentionally left as-is (a real NaN here validly means "no GitHub profile")

print("Remaining missing values:\n", df.isnull().sum())

## Step 9 — Remove Exact and Near-Duplicate Records

In [ ]:
before = len(df)

# Exact duplicates, ignoring the identifier column (two different IDs shouldn't count as unique
# if every other field is identical — that pattern indicates a re-submitted form)
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'student_id']).reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows -> new shape: {df.shape}")

# Near-duplicates: same student applying to the same opening more than once
near_dupes = df.duplicated(subset=['student_id', 'opening_id'], keep='first')
print("Near-duplicate (same student+opening) rows found:", near_dupes.sum())
df = df[~near_dupes].reset_index(drop=True)
print("Shape after removing near-duplicates:", df.shape)

## Step 10 — Detect and Treat Outliers (IQR Method)

In [ ]:
def cap_outliers_iqr(dataframe, column):
    q1, q3 = dataframe[column].quantile(0.25), dataframe[column].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((dataframe[column] < lower) | (dataframe[column] > upper)).sum()
    dataframe[column] = dataframe[column].clip(lower, upper)
    print(f"{column}: capped {n_outliers} outliers to [{lower:.2f}, {upper:.2f}]")
    return dataframe

for col in ['time_to_response_days', 'expected_stipend', 'match_score']:
    df = cap_outliers_iqr(df, col)

## Step 11 — Encode Categorical Data

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding for the ordinal-like status column
le = LabelEncoder()
df['status_label_encoded'] = le.fit_transform(df['status_label'])
print(dict(zip(le.classes_, le.transform(le.classes_))))

# One-Hot Encoding for nominal columns with no inherent order
df_encoded = pd.get_dummies(
    df, columns=['branch', 'city', 'opening_type', 'industry'], drop_first=True
)
print("Shape after encoding:", df_encoded.shape)

## Step 12 — Feature Engineering

In [ ]:
df_encoded['fit_efficiency'] = df_encoded['match_score'] / (df_encoded['skill_gap_count'] + 1)
df_encoded['is_high_stipend'] = (df_encoded['expected_stipend'] > df_encoded['expected_stipend'].median()).astype(int)

## Step 13 — Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ['cgpa', 'match_score', 'time_to_response_days', 'skill_gap_count',
            'expected_stipend', 'fit_efficiency']
scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

df_encoded[num_cols].agg(['mean', 'std']).round(2)

## Step 14 — Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

drop_cols = ['student_id', 'student_name', 'opening_id', 'opening_title', 'company_name',
             'required_skills', 'recruiter_designation', 'application_date', 'github_username',
             'status_label', 'status_label_encoded']
X = df_encoded.drop(columns=drop_cols)
y = df_encoded['status_label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

## Step 15 — Export the Cleaned Dataset

In [ ]:
df_encoded.to_csv('application_matching_preprocessed.csv', index=False)
from google.colab import files
files.download('application_matching_preprocessed.csv')